# Data harmonisation

The `unmapped entity` will be converted into `linked_eneties` with the link with other datasets (i.e. secondary, supplementary, and linked_entity).

The harmonisation process:
1. read the unmapped entities, one by one
2. link the unmapped entity one by one, by linking each key property (foreign id); when link to the dataset, check the existing data (using LLM/ollama)
    - if there are existing data (e.g. technology), then use the existing one
    - if no existing data, then create a few on
3. create a linked entity form the unmapped entity, fill with foreign key

### Import Libraries
This cell imports the necessary Python libraries for data manipulation and working with YAML files.

In [1]:
import os
import pandas as pd
import yaml

## Load data

### Load All Schema Definitions
This cell defines a function to recursively load all YAML schema files from the `../schema/` directory into a dictionary called `all_schemas`. These schemas are crucial for understanding the structure and validation rules for different data entities.

In [2]:
# Define the base directory for schema files
base_dir = '../schema/'

# Initialize an empty dictionary to hold all loaded schemas
all_schemas = {}

# Function to recursively load YAML files from a given directory
def load_yaml_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.yaml'):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    schema = yaml.safe_load(f)
                    all_schemas[file] = schema

# Call the function to load all YAML files from the schema directory and its subdirectories
load_yaml_files(base_dir)

# Print the number of loaded schemas for verification
print(f"Loaded {len(all_schemas)} schema files.")

Loaded 13 schema files.


### Load Unmapped Entities
This cell loads the unmapped entities from the `unmapped_entities_refuel.yaml` file, which will be used for harmonisation.

In [3]:
## load the unmapped entities from the yaml file
path = r'../motel-db\unmapped_entity\unmapped_entities_refuel.yaml'

with open(path, "r", encoding="utf-8") as f:
    ue = yaml.safe_load(f)

print(f"Unmapped entities: {len(ue)}")

Unmapped entities: 99


In [4]:
# test on the first 3 unmapped entities
ue = ue[:3]

### Load Existing Datasets
This cell defines a function to recursively load all CSV data files from the `../motel-db/` directory into a dictionary called `all_csv_data`. These datasets represent existing linked entities and controlled vocabularies that will be used during the harmonisation process.

In [5]:
# Define the base directory for data files
data_dir = '../motel-db/'

# Initialize an empty dictionary to hold all loaded CSV data
all_csv_data = {}

# Function to recursively load CSV files from a given directory
def load_csv_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                with open(file_path, mode='r', encoding='utf-8') as f:
                    try:
                        data = pd.read_csv(f)
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")
                        data = None
                        continue
                    # Use the filename (without path/extension) or full path as key
                    file_name = file.split('.')[0]  # or use file_path for full paSth
                    all_csv_data[file_name] = data

# Load all CSV files from the motel-db directory and its subdirectories
load_csv_files(data_dir)

# Print the number of loaded CSV files for verification
print(f"Loaded {len(all_csv_data)} CSV files.")

Loaded 14 CSV files.


# Harmonisation process

### Harmonisation Configuration and Helper Functions
This section defines the configuration for various entity types (technology, process, source, carrier) including their file paths, ID fields, prefixes, and columns.

It also includes helper functions for loading existing registries, appending new rows to CSV files, and an LLM-based function to resolve entities by matching candidates to existing registry entries or creating new ones if no match is found.

In [ ]:
import csv, json, os, re
from pathlib import Path
import ollama

In [ ]:
MODEL = "qwen3:14b"

# --- Config: entity type -> CSV path, id field, id prefix, columns, name field ---
ENTITY_CONFIG = {
    "technology": {
        "path": "../motel-db/secondary/technology.csv",
        "id_field": "tech_id", "prefix": "TECH", "name_field": "technology_name",
        "cols": ["tech_id", "technology_name", "technology_type", "technology_category", "technology_description"],
    },
    "process": {
        "path": "../motel-db/secondary/process.csv",
        "id_field": "process_id", "prefix": "PROC", "name_field": "process_name",
        "cols": ["process_id", "process_name", "process_type", "process_category", "process_description"],
    },
    "source": {
        "path": "../motel-db/secondary/source.csv",
        "id_field": "source_id", "prefix": "SRC", "name_field": "source_name",
        "cols": ["source_id", "source_name", "source_description", "source_type"],
    },
    "carrier": {
        "path": "../motel-db/controlled_vocabulary/carrier.csv",
        "id_field": "carrier_id", "prefix": "CAR", "name_field": "carrier_name",
        "cols": ["carrier_id", "carrier_name", "carrier_description", "carrier_type", "carrier_category"],
    },
}

# --- Registry helpers ---
def load_registry(entity_type):
    """
    Loads the existing registry for a given entity type from its corresponding CSV file.
    If the file does not exist, an empty list is returned.

    Args:
        entity_type (str): The type of entity (e.g., "technology", "process").

    Returns:
        list: A list of dictionaries, where each dictionary represents a row in the CSV.
    """
    path = ENTITY_CONFIG[entity_type]["path"]
    if not Path(path).exists():
        return []
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def append_row(entity_type, row):
    """
    Appends a new row to the CSV file corresponding to the given entity type.
    If the file or its directory does not exist, they will be created.
    The header is written only if the file is newly created.

    Args:
        entity_type (str): The type of entity.
        row (dict): A dictionary representing the row to append, with keys matching column names.
    """
    cfg = ENTITY_CONFIG[entity_type]
    Path(cfg["path"]).parent.mkdir(parents=True, exist_ok=True)
    file_exists = Path(cfg["path"]).exists()
    with open(cfg["path"], "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=cfg["cols"])
        if not file_exists:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in cfg["cols"]})

# --- LLM-based entity resolver ---
def resolve_entity(entity_type, candidate: dict, registry: list) -> str:
    """Match candidate to registry (exact then LLM), or create a new row. Returns resolved ID."""
    cfg = ENTITY_CONFIG[entity_type]
    id_field, name_field = cfg["id_field"], cfg["name_field"]
    candidate_name = str(candidate.get(name_field, "")).strip().lower()

    # Exact match first (fast, no LLM cost)
    for row in registry:
        if str(row.get(name_field, "")).strip().lower() == candidate_name:
            return row[id_field]

    # LLM semantic match when registry is non-empty
    # If a registry exists, use the LLM (ollama) to semantically match the candidate.
    # The prompt is carefully constructed to guide the LLM to respond with a JSON object
    # indicating whether a match was found and, if so, the ID of the existing entity.
    if registry:
        prompt = (
            f"Registry:\n{json.dumps(registry, indent=2)}\n\n"
            f"Candidate:\n{json.dumps(candidate, indent=2)}\n\n"
            f"Does the candidate semantically match any registry row?\n"
            f'Reply ONLY with JSON: {{"match": true, "id": "<existing_id>"}} or {{"match": false}}'
        )
        resp = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a data entity resolver. Output only valid JSON, no markdown or extra text."},
                {"role": "user", "content": prompt},
            ],
            options={"temperature": 0.0},
        )
        raw = re.sub(r"```(?:json)?|```", "", resp["message"]["content"]).strip()
        # strip ･･････ blocks if model uses chain-of-thought
        raw = re.sub(r"･･.*?･･", "", raw, flags=re.DOTALL).strip()
        decision = json.loads(raw)
        if decision.get("match"):
            return decision["id"]

    # Create new row
    # If no exact or semantic match is found, create a new entry in the registry.
    # A new ID is generated, the candidate's data is augmented with this ID,
    # and the new row is appended to both the CSV file and the in-memory registry.
    new_id = f"{cfg['prefix']}_{len(registry) + 1:05d}"
    new_row = {id_field: new_id, **candidate}
    append_row(entity_type, new_row)
    registry.append(new_row)
    print(f"  + {entity_type}: {new_id}  ({candidate.get(name_field)})")
    return new_id

print("Helpers ready.")

Helpers ready.


### Step 1: Collect Unique Candidates
This cell iterates through the unmapped entities and extracts unique candidates for technology, process, source, and carrier types. This ensures that each unique name is only resolved once, improving efficiency and consistency.

In [7]:
# --- Step 1: Collect unique candidates per entity type (dedup so each name is resolved once) ---

def collect_candidates(unmapped_entities):
    candidates = {et: {} for et in ENTITY_CONFIG}
    for e in unmapped_entities:
        t = e.get("technology", {})
        tname = e.get("technology_name")
        if tname:
            candidates["technology"].setdefault(tname, {
                "technology_name": tname,
                "technology_type": t.get("technology_type"),
                "technology_category": t.get("technology_category"),
                "technology_description": t.get("technology_description"),
            })
        pname = t.get("process_name")
        if pname:
            candidates["process"].setdefault(pname, {
                "process_name": pname,
                "process_type": t.get("process_type"),
                "process_category": t.get("process_category"),
            })
        for src in e.get("sources", []):
            sname = src.get("source_name")
            if sname:
                candidates["source"].setdefault(sname, {
                    "source_name": sname,
                    "source_description": src.get("source_description"),
                })
        for item in e.get("balancing", {}).get("inputs", []) + e.get("balancing", {}).get("outputs", []):
            cname = item.get("carrier_name")
            if cname:
                candidates["carrier"].setdefault(cname, {"carrier_name": cname})
    return candidates

candidates = collect_candidates(ue)
for et, items in candidates.items():
    print(f"  {et}: {len(items)} unique candidates")

# --- Step 2: Resolve all LLM-backed entities ---
registries = {et: load_registry(et) for et in ENTITY_CONFIG}
resolved_ids = {et: {} for et in ENTITY_CONFIG}  # {entity_type: {name -> id}}

for entity_type, entity_candidates in candidates.items():
    print(f"\nResolving {entity_type}...")
    registry = registries[entity_type]
    name_field = ENTITY_CONFIG[entity_type]["name_field"]
    for name, candidate in entity_candidates.items():
        resolved_ids[entity_type][name] = resolve_entity(entity_type, candidate, registry)

print("\nLLM resolution complete.")

  technology: 3 unique candidates
  process: 2 unique candidates
  source: 6 unique candidates
  carrier: 3 unique candidates

Resolving technology...

Resolving process...

Resolving source...

Resolving carrier...

LLM resolution complete.


In [8]:
# --- Step 3: Resolve controlled vocabulary (attributes + scopes) — deterministic, no LLM ---

SCOPE_CONFIG = {
    "geographic_scope": "../motel-db/controlled_vocabulary/geographic_scope.csv",
    "temporal_scope":   "../motel-db/controlled_vocabulary/temporal_scope.csv",
    "capacity_scope":   "../motel-db/controlled_vocabulary/capacity_scope.csv",
    "system_boundary":  "../motel-db/controlled_vocabulary/system_boundary.csv",
}
ATTR_PATH = "../motel-db/controlled_vocabulary/attribute.csv"
ATTR_COLS = ["attribute_id", "attribute_name", "attribute_description", "unit", "data_format"]

def load_attr_registry():
    if not Path(ATTR_PATH).exists():
        return {}
    with open(ATTR_PATH, encoding="utf-8") as f:
        return {r["attribute_name"]: r["attribute_id"] for r in csv.DictReader(f)}

def ensure_attr(name, description, registry):
    if name in registry:
        return registry[name]
    new_id = f"ATTR_{len(registry) + 1:05d}"
    registry[name] = new_id
    Path(ATTR_PATH).parent.mkdir(parents=True, exist_ok=True)
    file_exists = Path(ATTR_PATH).exists()
    with open(ATTR_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=ATTR_COLS)
        if not file_exists:
            writer.writeheader()
        writer.writerow({"attribute_id": new_id, "attribute_name": name,
                         "attribute_description": description or "", "unit": "", "data_format": ""})
    return new_id

def ensure_scope(scope_type, value):
    if not value:
        return None
    token = str(value).strip()
    path = SCOPE_CONFIG[scope_type]
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    existing_tokens = set()
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            existing_tokens = {r.get(scope_type, "").strip() for r in csv.DictReader(f)}
    if token not in existing_tokens:
        desc_field = f"{scope_type}_description"
        cols = [scope_type, desc_field, "note"]
        file_exists = Path(path).exists()
        with open(path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=cols)
            if not file_exists:
                writer.writeheader()
            writer.writerow({scope_type: token, desc_field: token, "note": ""})
    return token

attr_registry = load_attr_registry()
attr_ids = {}    # attribute_name -> attribute_id
scope_ids = {}   # (scope_type, value) -> token

for entity in ue:
    for attr in entity.get("attributes", []):
        name = attr.get("attribute_name")
        if name and name not in attr_ids:
            attr_ids[name] = ensure_attr(name, attr.get("notes", ""), attr_registry)
    scope = entity.get("scope", {})
    for scope_type in SCOPE_CONFIG:
        val = scope.get(f"{scope_type}_description")
        key = (scope_type, val)
        if val and key not in scope_ids:
            scope_ids[key] = ensure_scope(scope_type, val)

print(f"Attributes resolved: {len(attr_ids)}")
print(f"Scope tokens resolved: {len(scope_ids)}")

Attributes resolved: 13
Scope tokens resolved: 4


In [9]:
# --- Step 4: Build linked_entity records and save ---
from datetime import date

LE_PATH = "../motel-db/linked_entity/linked_entity.csv"
LE_COLS = [
    "linked_entity_id", "tech_id", "process_id",
    "geographic_scope", "temporal_scope", "capacity_scope", "system_boundary",
    "balancing_inputs", "balancing_outputs",
    "sources_json", "values_json",
    "date_created",
]

linked_entities = []
today = str(date.today())

for i, entity in enumerate(ue):
    t = entity.get("technology", {})
    scope = entity.get("scope", {})

    # Scope foreign keys
    geo  = scope_ids.get(("geographic_scope", scope.get("geographic_scope_description")), "")
    temp = scope_ids.get(("temporal_scope",   scope.get("temporal_scope_description")), "")
    cap  = scope_ids.get(("capacity_scope",   scope.get("capacity_scope_description")), "")
    sys_b = scope_ids.get(("system_boundary", scope.get("system_boundary_description")), "")

    # Balancing: carrier names -> carrier IDs
    bal = entity.get("balancing", {})
    inputs  = [{"carrier_id": resolved_ids["carrier"].get(x["carrier_name"], ""), "share": x.get("share"), "unit": x.get("unit")}
               for x in bal.get("inputs", []) if x.get("carrier_name")]
    outputs = [{"carrier_id": resolved_ids["carrier"].get(x["carrier_name"], ""), "share": x.get("share"), "unit": x.get("unit")}
               for x in bal.get("outputs", []) if x.get("carrier_name")]

    # Sources with resolved attribute IDs
    sources_resolved = [
        {"source_id": resolved_ids["source"].get(s["source_name"], ""),
         "linked_attribute_ids": [attr_ids.get(a, "") for a in s.get("linked_attribute", [])]}
        for s in entity.get("sources", []) if s.get("source_name")
    ]

    # Attribute values
    values = [
        {"attribute_id": attr_ids.get(a["attribute_name"], ""),
         "attribute_name": a["attribute_name"],
         "value": a.get("value"),
         "time_index": a.get("time_index")}
        for a in entity.get("attributes", []) if a.get("attribute_name")
    ]

    linked_entities.append({
        "linked_entity_id": f"LE_{i+1:05d}",
        "tech_id":          resolved_ids["technology"].get(entity.get("technology_name"), ""),
        "process_id":       resolved_ids["process"].get(t.get("process_name"), ""),
        "geographic_scope": geo,
        "temporal_scope":   temp,
        "capacity_scope":   cap,
        "system_boundary":  sys_b,
        "balancing_inputs":  json.dumps(inputs),
        "balancing_outputs": json.dumps(outputs),
        "sources_json":      json.dumps(sources_resolved),
        "values_json":       json.dumps(values),
        "date_created":      today,
    })

Path(LE_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(LE_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LE_COLS)
    writer.writeheader()
    writer.writerows(linked_entities)

print(f"Saved {len(linked_entities)} linked entities -> {LE_PATH}")

# Quick summary
import pandas as pd
df = pd.read_csv(LE_PATH)
print(df[["linked_entity_id", "tech_id", "process_id", "geographic_scope", "temporal_scope"]].head(10).to_string())

Saved 3 linked entities -> ../motel-db/linked_entity/linked_entity.csv
  linked_entity_id     tech_id  process_id geographic_scope  temporal_scope
0         LE_00001  TECH_00001  PROC_00002              ECA            2050
1         LE_00002  TECH_00002  PROC_00002              ECA            2050
2         LE_00003  TECH_00003  PROC_00002              ECA            2050


# Mapping files

**A** — flat provenance: one row per unmapped entity → linked entity  
**C** — per-entity-type lookup: original name → assigned ID + status (matched/created)

In [10]:
MAPPING_DIR = Path("../motel-db/mapping")
MAPPING_DIR.mkdir(parents=True, exist_ok=True)

# --- Mapping A: flat provenance (unmapped index -> linked entity) ---
MAP_A_PATH = MAPPING_DIR / "unmapped_to_linked.csv"
MAP_A_COLS = [
    "unmapped_index", "technology_name",
    "linked_entity_id", "tech_id", "process_id",
    "geographic_scope", "temporal_scope", "capacity_scope", "system_boundary",
    "source_ids", "date_mapped",
]

with open(MAP_A_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=MAP_A_COLS)
    writer.writeheader()
    for i, (entity, le) in enumerate(zip(ue, linked_entities)):
        source_ids = [s["source_id"] for s in json.loads(le["sources_json"])]
        writer.writerow({
            "unmapped_index":   i,
            "technology_name":  entity.get("technology_name", ""),
            "linked_entity_id": le["linked_entity_id"],
            "tech_id":          le["tech_id"],
            "process_id":       le["process_id"],
            "geographic_scope": le["geographic_scope"],
            "temporal_scope":   le["temporal_scope"],
            "capacity_scope":   le["capacity_scope"],
            "system_boundary":  le["system_boundary"],
            "source_ids":       json.dumps(source_ids),
            "date_mapped":      today,
        })

print(f"A: saved {len(linked_entities)} rows -> {MAP_A_PATH}")

# --- Mapping C: per-entity-type lookup (original name -> id + status) ---
_status = globals().get("resolution_status", {})  # graceful fallback if helpers weren't re-run

for entity_type, cfg in ENTITY_CONFIG.items():
    path = MAPPING_DIR / f"{entity_type}_map.csv"
    name_field = cfg["name_field"]
    id_field   = cfg["id_field"]
    registry   = load_registry(entity_type)
    name_to_id = {r[name_field]: r[id_field] for r in registry}

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[name_field, id_field, "status"])
        writer.writeheader()
        for name, rid in name_to_id.items():
            status = _status.get(entity_type, {}).get(name, "pre-existing")
            writer.writerow({name_field: name, id_field: rid, "status": status})
    print(f"C: {entity_type}_map.csv  ({len(name_to_id)} rows)")

# Attribute map
attr_path = MAPPING_DIR / "attribute_map.csv"
with open(attr_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["attribute_name", "attribute_id", "status"])
    writer.writeheader()
    for name, aid in attr_ids.items():
        writer.writerow({"attribute_name": name, "attribute_id": aid, "status": "created"})
print(f"C: attribute_map.csv  ({len(attr_ids)} rows)")

# Scope maps
for scope_type in SCOPE_CONFIG:
    path = MAPPING_DIR / f"{scope_type}_map.csv"
    scope_entries = {val: token for (st, val), token in scope_ids.items() if st == scope_type}
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["original_value", "scope_token", "status"])
        writer.writeheader()
        for val, token in scope_entries.items():
            writer.writerow({"original_value": val, "scope_token": token, "status": "created"})
    print(f"C: {scope_type}_map.csv  ({len(scope_entries)} rows)")

A: saved 3 rows -> ..\motel-db\mapping\unmapped_to_linked.csv
C: technology_map.csv  (41 rows)
C: process_map.csv  (2 rows)
C: source_map.csv  (6 rows)
C: carrier_map.csv  (3 rows)
C: attribute_map.csv  (13 rows)
C: geographic_scope_map.csv  (1 rows)
C: temporal_scope_map.csv  (1 rows)
C: capacity_scope_map.csv  (1 rows)
C: system_boundary_map.csv  (1 rows)


In [11]:
def generate_audit(indices=None):
    """
    On-demand B-style audit report. Joins mapping A + C + original YAML.
    Pass indices=None for all, or a list like [0, 5, 12] for spot-checks.
    """
    with open(MAP_A_PATH, encoding="utf-8") as f:
        map_a = {int(r["unmapped_index"]): r for r in csv.DictReader(f)}

    c_maps = {}
    for et, cfg in ENTITY_CONFIG.items():
        path = MAPPING_DIR / f"{et}_map.csv"
        if path.exists():
            with open(path, encoding="utf-8") as f:
                c_maps[et] = {r[cfg["name_field"]]: r for r in csv.DictReader(f)}

    scope_c = {}
    for scope_type in SCOPE_CONFIG:
        path = MAPPING_DIR / f"{scope_type}_map.csv"
        if path.exists():
            with open(path, encoding="utf-8") as f:
                scope_c[scope_type] = {r["original_value"]: r["scope_token"] for r in csv.DictReader(f)}

    targets = indices if indices is not None else list(map_a.keys())
    report = []

    for idx in targets:
        entity = ue[idx]
        a = map_a.get(idx, {})
        t = entity.get("technology", {})

        report.append({
            "unmapped_index":   idx,
            "technology_name":  entity.get("technology_name"),
            "linked_entity_id": a.get("linked_entity_id"),
            "resolution": {
                "technology": c_maps.get("technology", {}).get(entity.get("technology_name"), {}),
                "process":    c_maps.get("process", {}).get(t.get("process_name"), {}),
                "sources":    [c_maps.get("source", {}).get(s["source_name"], {}) for s in entity.get("sources", [])],
                "carriers": {
                    "inputs":  [c_maps.get("carrier", {}).get(x["carrier_name"], {}) for x in entity.get("balancing", {}).get("inputs", [])],
                    "outputs": [c_maps.get("carrier", {}).get(x["carrier_name"], {}) for x in entity.get("balancing", {}).get("outputs", [])],
                },
                "scope": {st: scope_c.get(st, {}).get(entity.get("scope", {}).get(f"{st}_description")) for st in SCOPE_CONFIG},
            },
            "unresolved_attributes": [a["attribute_name"] for a in entity.get("attributes", []) if not attr_ids.get(a["attribute_name"])],
        })

    return report

# Example: audit the first 3 entities
for entry in generate_audit([0, 1, 2]):
    print(json.dumps(entry, indent=2))

{
  "unmapped_index": 0,
  "technology_name": "NH3_CCGT_El",
  "linked_entity_id": "LE_00001",
  "resolution": {
    "technology": {
      "technology_name": "NH3_CCGT_El",
      "tech_id": "TECH_00001",
      "status": "pre-existing"
    },
    "process": {
      "process_name": "CCGT",
      "process_id": "PROC_00002",
      "status": "pre-existing"
    },
    "sources": [
      {
        "source_name": "report_power_to_ammonia_2018",
        "source_id": "SRC_00006",
        "status": "pre-existing"
      },
      {
        "source_name": "conversion_parametrization",
        "source_id": "SRC_00002",
        "status": "pre-existing"
      },
      {
        "source_name": "report_danish_energy_2025",
        "source_id": "SRC_00003",
        "status": "pre-existing"
      }
    ],
    "carriers": {
      "inputs": [
        {
          "carrier_name": "NH3",
          "carrier_id": "CAR_00003",
          "status": "pre-existing"
        }
      ],
      "outputs": [
        {
     